In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
!pip install ultralytics
from ultralytics import YOLO

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

Cloning into 'yolov5'...
remote: Enumerating objects: 17972, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 17972 (delta 91), reused 31 (delta 31), pack-reused 17856 (from 3)
Receiving objects: 100% (17972/17972), 17.10 MiB | 17.25 MiB/s, done.
Resolving deltas: 100% (12226/12226), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [ ]:
import zipfile
import os
import shutil

# Define paths
drive_dataset_path = "/content/drive/MyDrive/brain tumor 2.v3i.yolov8.zip"
local_dataset_path = "/content/dataset"

# Create the local dataset directory if it doesn't exist
os.makedirs(local_dataset_path, exist_ok=True)

# Copy and extract dataset from Drive to local
print("📁 Copying and extracting dataset from Drive to local...")
if os.path.exists(drive_dataset_path):
    try:
        with zipfile.ZipFile(drive_dataset_path, 'r') as zip_ref:
            zip_ref.extractall(local_dataset_path)
        print("✅ Dataset extracted successfully!")

        # Check what we have
        print("\n📂 Dataset structure:")
        for item in os.listdir(local_dataset_path):
            item_path = os.path.join(local_dataset_path, item)
            if os.path.isdir(item_path):
                file_count = len(os.listdir(item_path))
                print(f"📁 {item}/ - {file_count} items")
            else:
                print(f"📄 {item}")

    except zipfile.BadZipFile:
        print("❌ Error: The file is not a valid zip archive.")
    except Exception as e:
        print(f"❌ An unexpected error occurred during extraction: {e}")
else:
    print("❌ Dataset ZIP file not found in Drive!")


📁 Copying and extracting dataset from Drive to local...
✅ Dataset extracted successfully!

📂 Dataset structure:
📁 test/ - 2 items
📄 data.yaml
📁 train/ - 2 items
📄 README.roboflow.txt
📄 README.dataset.txt
📁 valid/ - 2 items


In [ ]:
# Train YOLOv8n, YOLOv8s, YOLOv8m
models = ['yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt']

for model_name in models:
    print(f"\n Training {model_name}...")

    model = YOLO(model_name)

    model.train(
        data=f'{local_dataset_path}/data.yaml',  # Uses your train/valid splits
        epochs=50,
        imgsz=640,
        batch=16,
        name=f"{model_name.replace('.pt', '')}_brain",
        save=True
    )

    print(f"✅ {model_name} training completed!")


 Training yolov8n.pt...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_brain, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ov

In [ ]:
import os
from ultralytics import YOLO
import pandas as pd

results = {}

base_path = "/content/yolov5/runs/detect"

for folder in os.listdir(base_path):
    weight_path = os.path.join(base_path, folder, "weights", "best.pt")

    if os.path.exists(weight_path):
        print(f"🔍 Evaluating {folder}...")
        model = YOLO(weight_path)
        metrics = model.val()

        results[folder] = {
            "mAP50": metrics.box.map50,
            "mAP50-95": metrics.box.map,
            "Precision": metrics.box.mp,
            "Recall": metrics.box.mr,
            "Inference_Time_ms": sum(metrics.speed.values()) / len(metrics.speed),
            "Model_Size_MB": os.path.getsize(weight_path) / (1024 * 1024)
        }

comparison_df = pd.DataFrame(results).T
comparison_df

🔍 Evaluating yolov8s_brain...
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1229.0±388.3 MB/s, size: 33.9 KB)
val: Scanning /content/dataset/valid/labels.cache... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 6.5Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 10, len(boxes) = 20. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.8it/s 1.1s
                   all         20         20      0.929      0.892      0.968      0.683
                 Tumor         10         10          1      0.784      0.941      0.558
              no tum

,mAP50,mAP50-95,Precision,Recall,Inference_Time_ms,Model_Size_MB
yolov8s_brain,0.967778,0.682778,0.928570,0.891816,9.091967,21.475870
yolov8m_brain,0.936685,0.685927,0.886243,0.877652,9.904810,49.621843
yolov8n_brain,0.963590,0.716547,0.902199,0.865073,4.768428,5.960184
